# Blue Core API: every endpoint
One call to each endpoint of the Blue Core API, grouped by purpose, with the response it gives. Copy the cell you need into your own notebook. The last cell checks the list against the API's live OpenAPI spec, so an endpoint added later shows up as not yet covered.

| Purpose | Endpoints |
|---|---|
| About the service | `/`, `/health`, `/context.jsonld` |
| Find | `/search/`, `/search/profile` |
| Read records | hubs, works and instances, in each format; other resources; profiles |
| History | `…/versions`, `…/version/{id}`, change document feeds |
| Write | create, update and delete for each kind |
| Convert | `/marc2xml`, `/marc2bibframe` |
| Workflows | `/batches/`, `/batches/upload/`, `/export/` |

On your local stack it creates its own example records (marked `example.org/notebook/<run>`), uses them, and deletes them at the end. On dev or stage it reads existing records instead and skips every call that needs a login.

In [ ]:
import json, uuid
import pandas as pd
from bluecore import NotConfigured, marc21, show, target

bc = target()
WRITABLE = not bc.read_only
AUTH = bc.auth_headers() if WRITABLE else {}
LD = {**AUTH, "Content-Type": "application/ld+json"}
RUN = uuid.uuid4().hex[:8]
IDS = {}       # the records the calls use: hub_uuid, work_uuid, ...
CALLS = []     # every call made, for the coverage check at the end

def call(method, endpoint, *, suffix="", chars=600, **kwargs):
    """Calls an endpoint by its OpenAPI path, e.g. call("GET", "/works/{work_uuid}"), filling in IDS"""
    r = bc.http.request(method, "/api" + endpoint.format(**IDS) + suffix, **kwargs)
    CALLS.append({"method": method, "endpoint": endpoint, "status": r.status_code})
    show(r, chars)
    return r

def skip(method, endpoint, why="needs a login, so it only runs on your local stack"):
    CALLS.append({"method": method, "endpoint": endpoint, "status": "skipped: " + why})

bc

## About the service

In [ ]:
call("GET", "/")
call("GET", "/health")
call("GET", "/context.jsonld", chars=300);

## Example records
Locally: create a hub, a work and an instance of it, an other resource and a profile, from the request examples the API documents (`bc.example`). On dev or stage: pick existing ones.

In [ ]:
if WRITABLE:
    hub = {**bc.example("/hubs/"), "@id": f"https://example.org/notebook/{RUN}/hub"}
    IDS["hub_uuid"] = call("POST", "/hubs/", json=hub, headers=LD, chars=300).json()["uuid"]

    work = {**bc.example("/works/"), "@id": f"https://example.org/notebook/{RUN}/work"}
    created = call("POST", "/works/", json=work, headers=LD, chars=300).json()
    IDS["work_uuid"], work_id = created["uuid"], created["id"]

    # An instance of that work: the Sinopia body takes the work's database id
    instance = {**bc.example("/instances/"), "@id": f"https://example.org/notebook/{RUN}/instance"}
    body = {"data": json.dumps(instance), "work_id": work_id}
    IDS["instance_uuid"] = call("POST", "/instances/", json=body, headers={**AUTH, "Content-Type": "application/vnd.sinopia+json"}, chars=300).json()["uuid"]

    agent = f"https://example.org/notebook/{RUN}/agent"
    body = {"uri": agent, "data": json.dumps({"@id": agent, "rdfs:label": "Example agent"})}
    IDS["resource_id"] = call("POST", "/resources/", json=body, headers=AUTH, chars=300).json()["id"]

    template = [{"@id": "", "@type": ["http://sinopia.io/vocabulary/ResourceTemplate"],
                 "http://sinopia.io/vocabulary/hasResourceId": [{"@value": f"notebook:example:{RUN}"}],
                 "http://www.w3.org/2000/01/rdf-schema#label": [{"@value": "Notebook example profile"}]}]
    IDS["profile_uuid"] = call("POST", "/profiles/", json={"data": json.dumps(template)}, headers=AUTH, chars=300).json()["uuid"]
else:
    for method, endpoint in [("POST", "/hubs/"), ("POST", "/works/"), ("POST", "/instances/"), ("POST", "/resources/"), ("POST", "/profiles/")]:
        skip(method, endpoint)
    IDS["hub_uuid"], IDS["work_uuid"], IDS["instance_uuid"] = bc.sample("hubs"), bc.sample("works"), bc.sample("instances")
    IDS["resource_id"] = bc.get("/api/resources/", params={"limit": 1}).json()["resources"][0]["id"]
    IDS["profile_uuid"] = bc.get("/api/profiles/", params={"limit": 1}).json()["profiles"][0]["uuid"]
IDS

## Find
Full-text search over hubs, works and instances (`type`: `all`, `hubs`, `works`, `instances`; `scope`: `all` or `title`; quotes for a phrase), and over profiles (`nested`: `include` or `exclude`).

In [ ]:
call("GET", "/search/", params={"q": "pride and prejudice", "type": "all", "limit": 2}, chars=500)
call("GET", "/search/profile", params={"q": "title", "limit": 1}, chars=500);

## Read records
Hubs, works and instances negotiate their format: a suffix (`.jsonld`, `.ttl`, `.nt`, `.rdf`, `.cbd.jsonld`, `.cbd.xml`, `.vnd.sinopia.json`) or an `Accept` header. With neither, they answer with the HTML page people see. `expand=true` adds the linked resources' data.

In [ ]:
call("GET", "/hubs/{hub_uuid}", suffix=".jsonld", chars=400)
call("GET", "/works/{work_uuid}", suffix=".ttl", chars=400)
call("GET", "/works/{work_uuid}", params={"expand": "true"}, headers={"Accept": "application/ld+json"}, chars=300)
call("GET", "/instances/{instance_uuid}", suffix=".cbd.jsonld", chars=400)
print("No suffix or Accept header:", bc.get(f"/api/instances/{IDS['instance_uuid']}").headers["content-type"])

In [ ]:
call("GET", "/resources/", params={"limit": 1}, chars=400)
call("GET", "/resources/{resource_id}", chars=400)
call("GET", "/profiles/", params={"limit": 1}, chars=300)
call("GET", "/profiles/{profile_uuid}", chars=300);

## Edit
A PUT sends the whole record again. Locally: change each example record's label or title.

In [ ]:
if WRITABLE:
    hub = bc.get(f"/api/hubs/{IDS['hub_uuid']}.jsonld").json()
    hub["rdfs:label"] = ["Austen, Jane, 1775-1817. Pride and prejudice (edited)"]
    call("PUT", "/hubs/{hub_uuid}", json=hub, headers=LD, chars=200)

    work = bc.get(f"/api/works/{IDS['work_uuid']}.jsonld").json()
    work["title"] = [{"@type": ["Title"], "mainTitle": ["Pride and prejudice (edited)"]}]
    call("PUT", "/works/{work_uuid}", json=work, headers=LD, chars=200)

    instance = bc.get(f"/api/instances/{IDS['instance_uuid']}.jsonld").json()
    instance["rdfs:label"] = ["Pride and prejudice (Penguin Classics, 2003, edited)"]
    call("PUT", "/instances/{instance_uuid}", json=instance, headers=LD, chars=200)

    agent = f"https://example.org/notebook/{RUN}/agent"
    call("PUT", "/resources/{resource_id}", json={"data": json.dumps({"@id": agent, "rdfs:label": "Example agent (edited)"})}, headers=AUTH, chars=200)

    template[0]["http://www.w3.org/2000/01/rdf-schema#label"] = [{"@value": "Notebook example profile (edited)"}]
    call("PUT", "/profiles/{profile_uuid}", json={"data": json.dumps(template)}, headers=AUTH, chars=200)
else:
    for kind in ["/hubs/{hub_uuid}", "/works/{work_uuid}", "/instances/{instance_uuid}", "/resources/{resource_id}", "/profiles/{profile_uuid}"]:
        skip("PUT", kind)

## History
Every save makes a version. `…/versions` lists them, `…/version/{id}` returns the record as it was. The change document feeds (Activity Streams) list every create and update, oldest first, a page at a time.

> **Known issue:** an empty feed names `page/0` as its last page, and that page answers `500` (a negative SQL offset). <!-- found 2026-10-16, bluecore_api 0.35.2 -->

In [ ]:
for kind, key in [("hubs", "hub_uuid"), ("works", "work_uuid"), ("instances", "instance_uuid"), ("profiles", "profile_uuid")]:
    versions = call("GET", f"/{kind}/{{{key}}}/versions", chars=300).json()["versions"]
    IDS["version_id"] = versions[0]["id"]
    call("GET", f"/{kind}/{{{key}}}/version/{{version_id}}", chars=200)

In [ ]:
for kind in ["works", "instances"]:
    feed = call("GET", f"/change_documents/{kind}/feed", chars=400).json()
    IDS["id"] = max(int(feed["last"]["id"].rsplit("/", 1)[1]), 1)   # the newest page (an empty feed's "last" is page 0, which fails)
    call("GET", f"/change_documents/{kind}/page/{{id}}", chars=400)

## Embeddings
Reserved for vector embeddings: the API answers `501 Not implemented` for now.

In [ ]:
for kind, key in [("hubs", "hub_uuid"), ("works", "work_uuid"), ("instances", "instance_uuid")]:
    call("GET", f"/{kind}/{{{key}}}/embeddings", chars=200)
    if WRITABLE:
        call("POST", f"/{kind}/{{{key}}}/embeddings", headers=AUTH, chars=200)
    else:
        skip("POST", f"/{kind}/{{{key}}}/embeddings")

## Convert
One MARC record at a time, in the API itself (no Airflow). `/marc2xml` takes binary MARC21; `/marc2bibframe` takes MARCXML or binary MARC and answers in the format you `Accept` (JSON-LD by default, Turtle, RDF/XML, N-Triples). `marc21()` builds a small binary record for testing.

In [ ]:
if WRITABLE:
    record = marc21([
        ("001", "92005291"),
        ("008", "920219s1993    caua          001 0 eng  "),
        ("100", "1 ", [("a", "Austen, Jane,"), ("d", "1775-1817.")]),
        ("245", "10", [("a", "Pride and prejudice /"), ("c", "Jane Austen.")]),
    ])
    marcxml = call("POST", "/marc2xml", data=record, headers={**AUTH, "Content-Type": "application/marc"}, chars=400).text
    call("POST", "/marc2bibframe", data=marcxml.encode(), headers={**AUTH, "Content-Type": "application/xml", "Accept": "text/turtle"}, chars=500)
else:
    skip("POST", "/marc2xml")
    skip("POST", "/marc2bibframe")

## Workflows
These start Airflow DAGs. Here, `/batches/upload/` loads the example work through the `resource_loader` DAG for real, then deletes what it loaded. `/batches/` (load from a URI) and `/export/` (send an instance to a library catalog) are only probed with an empty body, which they refuse (`422`), so nothing runs. The **Load records with a batch** and **Export to a catalog** starters run them properly.

In [ ]:
if WRITABLE:
    work = {**bc.example("/works/"), "@id": f"https://example.org/notebook/{RUN}/batch-work", "rdfs:label": f"Notebook batch {RUN}"}
    batch = call("POST", "/batches/upload/", files={"file": (f"notebook-{RUN}.jsonld", json.dumps(work), "application/ld+json")}, headers=AUTH).json()
    state, tasks = bc.wait_for_run("resource_loader", batch["workflow_id"])
    print("resource_loader:", state, tasks)
    loaded = [x for x in bc.get("/api/search/", params={"q": f"Notebook batch {RUN}", "type": "works"}).json()["results"]]
    IDS["batch_work_uuids"] = [x["uuid"] for x in loaded]

    call("POST", "/batches/", json={}, headers=AUTH, chars=200)
    call("POST", "/export/", json={}, headers=AUTH, chars=300)
else:
    for endpoint in ["/batches/upload/", "/batches/", "/export/"]:
        skip("POST", endpoint)

## Delete
Locally: delete everything this notebook made (instances before their work). The vocabulary terms the records pointed at (`id.loc.gov/vocabulary/...`) stay, as other resources: the API keeps them for other records.

In [ ]:
if WRITABLE:
    call("DELETE", "/instances/{instance_uuid}", headers=AUTH)
    call("DELETE", "/works/{work_uuid}", headers=AUTH)
    call("DELETE", "/hubs/{hub_uuid}", headers=AUTH)
    call("DELETE", "/resources/{resource_id}", headers=AUTH)
    call("DELETE", "/profiles/{profile_uuid}", headers=AUTH)
    for u in IDS.get("batch_work_uuids", []):
        print("batch work", u, bc.delete(f"/api/works/{u}", headers=AUTH).status_code)
else:
    for endpoint in ["/instances/{instance_uuid}", "/works/{work_uuid}", "/hubs/{hub_uuid}", "/resources/{resource_id}", "/profiles/{profile_uuid}"]:
        skip("DELETE", endpoint)

## Coverage
Every endpoint in the live OpenAPI spec, with what this notebook got back. One marked **not covered** is new: add a call for it above.

In [ ]:
spec = bc.openapi()
done = pd.DataFrame(CALLS).groupby(["method", "endpoint"])["status"].agg(lambda s: ", ".join(sorted({str(x) for x in s})))
coverage = pd.DataFrame([
    {"tag": (op.get("tags") or ["(none)"])[0], "method": m.upper(), "endpoint": path, "summary": op.get("summary"),
     "result": done.get((m.upper(), path), "not covered")}
    for path, ops in spec["paths"].items() for m, op in ops.items() if m in {"get", "post", "put", "delete", "patch"}
])
print(f"{(coverage['result'] != 'not covered').sum()} of {len(coverage)} endpoints called")
coverage.sort_values(["tag", "endpoint", "method"]).reset_index(drop=True)